In [1]:
import os
import math
import random
from typing import List, Dict
from dataclasses import dataclass

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoConfig,
)
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download("punkt", quiet=True)

[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1028)>


False

In [3]:
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

@dataclass
class Config:
    # Models (you can replace names if not available)
    code_model_name: str = "microsoft/CodeGPT-small-py"      # suggested code-specialized (GPT-like)
    general_model_name: str = "EleutherAI/gpt-neo-125M"      # suggested general-purpose
    code_classifier_name: str = "microsoft/codebert-base"    # code encoder for classification
    general_classifier_name: str = "bert-base-uncased"       # general encoder baseline

    # Data / training
    max_seq_len: int = 512
    gen_max_new_tokens: int = 128
    gen_train_samples: int = 2000     # reduce for quick runs; increase for serious runs
    gen_val_samples: int = 200
    cls_train_samples: int = 2000
    cls_val_samples: int = 400
    batch_size: int = 4                # keep small if GPU memory limited
    cls_batch_size: int = 16
    num_epochs_gen: int = 2
    num_epochs_cls: int = 3
    lr: float = 5e-5
    output_dir: str = "./results_llm_lab"

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)

Device: cpu


In [4]:
def normalize_code_text(s: str) -> str:
    """Minimal normalization for code comparison: strip trailing spaces, collapse multi-blank lines."""
    # keep indentation as it's meaningful for Python; just strip trailing whitespace lines
    lines = [ln.rstrip() for ln in s.splitlines()]
    # remove leading/trailing blank lines
    while lines and lines[0] == "":
        lines.pop(0)
    while lines and lines[-1] == "":
        lines.pop(-1)
    return "\n".join(lines)